In [ ]:
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from dataclasses import dataclass, InitVar, field, asdict
from pathlib import Path
import sys

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "project_paths.py").is_file():
        sys.path.insert(0, str(_p))
        break
from project_paths import data_dir, artifacts_lab6

DATA_DIR = data_dir()
ART_LAB6 = artifacts_lab6()

@dataclass
class MNIST:
    root: InitVar[str]
    _0: list[object] = field(init=False, default_factory=list)
    _1: list[object] = field(init=False, default_factory=list)
    _2: list[object] = field(init=False, default_factory=list)
    _3: list[object] = field(init=False, default_factory=list)
    _4: list[object] = field(init=False, default_factory=list)
    _5: list[object] = field(init=False, default_factory=list)
    _6: list[object] = field(init=False, default_factory=list)
    _7: list[object] = field(init=False, default_factory=list)
    _8: list[object] = field(init=False, default_factory=list)
    _9: list[object] = field(init=False, default_factory=list)

    def __post_init__(self, root):
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ])
        train_data = datasets.MNIST(
            root=root,
            train=True,
            download=True,
            transform=transform,
        )
        test_data = datasets.MNIST(
            root=root,
            train=False,
            download=True,
            transform=transform,
        )
        for image, label in train_data+test_data:
            getattr(self, f"_{label}").append(image)

    @classmethod
    def _nested_format(cls, obj, fmt, indent=0):
        sp = "\t" * indent

        if isinstance(obj, list):
            if not obj:
                return "{}"
            if not isinstance(obj[0], list):
                return "{" + ", ".join(fmt(x) for x in obj) + "}"
            inner = tuple(cls._nested_format(x, fmt, indent+1) for x in obj)
            return "{\n" + ",\n".join("\t" * (indent+1) + s for s in inner) + "\n" + sp + "}"
        else:
            return fmt(obj)

    def export_h(self, num, ctype, fmt, output_path):
        name = "IMAGES"
        nested = [
            getattr(self, f"_{i%10}")[i//10].flatten().tolist() for i in range(num)
        ]
        lines = [
            f"#pragma once",
            f"#include <cstddef>",
            f"",
            f"// {name} shape=({num}, 784), dtype={ctype}",
            f"#define IMAGE_NUM {num}",
            f"static const {ctype} {name}[{num}][784] = {self._nested_format(nested, fmt, 0)};",
        ]
        with open(output_path, 'w', encoding='utf-8') as h:
            h.write('\n'.join(lines))

    def export_np(self, output_path, mode):
        total = []
        for i in range(10):
            data = getattr(self, f"_{i}")
            
            if mode == "image":
                total.extend(data)
            else:
                total.extend([i for _ in data])

        np.save(output_path, np.array(total))

mnist = MNIST(str(DATA_DIR))

In [2]:
mnist.export_h(280, "float", lambda x: f"{repr(x)}f", str(ART_LAB6 / "images.h"))

In [3]:
mnist.export_np(str(ART_LAB6 / "MNIST_DATASET_IMAGE.npy"), 'image')
mnist.export_np(str(ART_LAB6 / "MNIST_DATASET_LABEL.npy"), 'label')

In [4]:
# test
test = np.load(str(ART_LAB6 / "MNIST_DATASET_LABEL.npy"), allow_pickle=True)
test

array([0, 0, 0, ..., 9, 9, 9], shape=(70000,))